In [66]:
from nltk.corpus import stopwords

#stopwords.words('english')

In [67]:
from sqlalchemy import create_engine
import pandas as pd
import re
import string
import nltk
pd.set_option('display.max_colwidth', 100)

In [68]:
QUERY=""" 
SELECT
trim(nvl(bm.name,'') || ' ' || nvl(description,'')) "feature"
,category_name "label"
FROM oneflare_reports.business_master bm
WHERE bm.total_quotes_made >= 3 and category_name is not null
"""

In [69]:
def make_engine():
    user = 'xxx'
    pw = 'xxx'
    db_name = 'snowplow'
    host = 'redshift.oneflare.io'
    port = 5439
    engine_string = "postgresql://%s:%s@%s:%s/%s" % (
        user, pw, host, port, db_name)
    engine = create_engine(engine_string)
    return engine

redshift = make_engine()
data=pd.read_sql(QUERY,redshift)


In [70]:
data.head()

,feature,label
0,"Abbey Road Constructions We, since 30 years have been working to provide workmanship services to...",Decking
1,A & E Accountants Pty Ltd A & E Accountants Pty Ltd is an established business providing Complet...,Accountant
2,"Walway Plastering No job too big or small.\nI have experience in painting, plastering, tiling.\n...",Plasterer
3,"Ampleamp 27 years as an electrician,running Ampleamp for last 7 years. Lots of experience in the...",Security System
4,First Class Cleaning Services 1st Class Commercial Cleaners Sydney provide superior cleaning ser...,Gardener


In [71]:
stopwords = nltk.corpus.stopwords.words('english')
wn = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()

In [93]:
def clean_text_by_porter_stem(text):

    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [ps.stem(word) for word in tokens if (word not in stopwords) and (word.isdigit() ==False) and (bool(re.match('^[a-zA-Z]+$',word))==True ) ]
    text = [ word for word in text if word in import_features_list]
    return text

def clean_text_by_wordnet_lemmatize(text):

    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [wn.lemmatize(word)  for word in tokens if (word not in stopwords) and (word.isdigit() ==False) and (bool(re.match('^[a-zA-Z]+$',word))==True )] 
    text = [ word for word in text if word in import_features_list]
    
    return text

In [13]:
#data.drop('feature_cleaned', axis=1, inplace=True)

In [28]:
data['feature_cleaned_ps']=data['feature'].apply(lambda x: clean_text_by_porter_stem( ('' if x is None else x).lower()))
data['feature_cleaned_wd']=data['feature'].apply(lambda x: clean_text_by_wordnet_lemmatize( ('' if x is None else x).lower()))

In [29]:
data.head()

,feature,label,feature_cleaned_ps,feature_cleaned_wd
0,"Anaxim Accounting Solutions The company is a qualified, professional accountant who deals with a...",Accountant,"[anaxim, account, solut, compani, qualifi, profession, account, deal, array, account, issu, prov...","[anaxim, accounting, solution, company, qualified, professional, accountant, deal, array, accoun..."
1,RCS Websites Let RCS Websites breath life in to your existing website!\r\nOur websites are simpl...,SEO and SEM,"[rc, websit, let, rc, websit, breath, life, exist, websit, websit, simpl, clean, profession, sur...","[rcs, website, let, rcs, website, breath, life, existing, website, website, simple, clean, profe..."
2,Intouch Carpet Cleaning Services InTouch Carpet and Pest Control is a family owned business who ...,Pest Control,"[intouch, carpet, clean, servic, intouch, carpet, pest, control, famili, own, busi, valu, old, f...","[intouch, carpet, cleaning, service, intouch, carpet, pest, control, family, owned, business, va..."
3,Shannon Thomas Carpenter Licensed Carpenter\r\n - Small Decks and Pergolas as well as re deck...,Carpenter,"[shannon, thoma, carpent, licens, carpent, small, deck, pergola, well, deck, repair, attent, det...","[shannon, thomas, carpenter, licensed, carpenter, small, deck, pergola, well, decking, repair, a..."
4,"M&M Cabinets Custom-made Cabinets for kitchen, laundries, bathroom and anywhere in your house. 3...",Builder,"[mm, cabinet, custommad, cabinet, kitchen, laundri, bathroom, anywher, hous, 3d, comput, draw, a...","[mm, cabinet, custommade, cabinet, kitchen, laundry, bathroom, anywhere, house, 3d, computer, dr..."


In [31]:
print(type(data.iloc[0]['feature_cleaned_ps']))
print(type(data.iloc[0]['feature_cleaned_wd']))

<class 'list'>
<class 'list'>


In [74]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [94]:
tfidf_vectorizer_ps = TfidfVectorizer(analyzer=clean_text_by_porter_stem)
tfidf_vectorized_matrix_ps = tfidf_vectorizer_ps.fit_transform(data['feature'] )



In [11]:
count_vectorizer_ps = CountVectorizer(analyzer=clean_text_by_porter_stem)
count_vectorized_matrix_ps = count_vectorizer_ps.fit_transform(data['feature'] )

In [95]:
tfidf_vectorized_matrix_ps.shape

(39025, 38077)

In [99]:
tfidf_vectorized_df_ps = pd.DataFrame(tfidf_vectorized_matrix_ps.toarray())

MemoryError: 

In [96]:
tfidf_vectorized_df_ps.head()

NameError: name 'tfidf_vectorized_df_ps' is not defined

In [9]:
tfidf_vectorizer_wn = TfidfVectorizer(analyzer=clean_text_by_wordnet_lemmatize)
tfidf_vectorized_matrix_wn = tfidf_vectorizer_wn.fit_transform(data['feature'])

In [10]:
tfidf_vectorized_matrix_wn.shape

(38974, 56825)

In [ ]:
tfidf_vectorizer_wn = TfidfVectorizer(analyzer=clean_text_by_wordnet_lemmatize)
tfidf_vectorized_matrix_wn = tfidf_vectorizer_wn.fit_transform(data['feature'])
tfidf_vectorized_df_wn = pd.DataFrame(tfidf_vectorized_matrix_wn.toarray())

In [97]:
tfidf_vectorized_df_ps_h = tfidf_vectorized_df_ps
tfidf_vectorized_df_ps_h.columns = tfidf_vectorizer_ps.get_feature_names()
tfidf_vectorized_df_ps_h.head()

NameError: name 'tfidf_vectorized_df_ps' is not defined

In [ ]:
tfidf_vectorized_df_wn_h = tfidf_vectorized_df_wn
tfidf_vectorized_df_wn_h.columns = tfidf_vectorizer_wn.get_feature_names()
tfidf_vectorized_df_wn_h.head()

In [16]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_fscore_support as score
import time

In [65]:
print(dir(RandomForestClassifier))
print(RandomForestClassifier())

['__abstractmethods__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_abc_cache', '_abc_negative_cache', '_abc_negative_cache_version', '_abc_registry', '_estimator_type', '_get_param_names', '_make_estimator', '_set_oob_score', '_validate_X_predict', '_validate_estimator', '_validate_y_class_weight', 'apply', 'decision_path', 'feature_importances_', 'fit', 'get_params', 'predict', 'predict_log_proba', 'predict_proba', 'score', 'set_params']
RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
            max_depth=None, max_features='auto', max_leaf_nodes=None,
            min_impurity_decrease=0.0,

In [66]:
print(dir(GradientBoostingClassifier))
print(GradientBoostingClassifier())

['_SUPPORTED_LOSS', '__abstractmethods__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_abc_cache', '_abc_negative_cache', '_abc_negative_cache_version', '_abc_registry', '_check_initialized', '_check_params', '_clear_state', '_decision_function', '_estimator_type', '_fit_stage', '_fit_stages', '_get_param_names', '_init_decision_function', '_init_state', '_is_initialized', '_make_estimator', '_resize_state', '_staged_decision_function', '_validate_estimator', '_validate_y', 'apply', 'decision_function', 'feature_importances_', 'fit', 'get_params', 'n_features', 'predict', 'predict_log_proba', 'predict_proba', 'score', 's

## Start to evaluate models

In [100]:
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.model_selection import train_test_split

In [101]:
X_train, X_test, y_train, y_test = train_test_split(tfidf_vectorized_matrix_ps, data['label'], test_size=0.2)

In [28]:
print(X_train)
print(y_train)

  (0, 12554)	0.897560979187
  (0, 43683)	0.418577057734
  (0, 38357)	0.138482978662
  (1, 546)	0.153514794957
  (1, 25221)	0.140580537403
  (1, 27985)	0.614059179828
  (1, 30343)	0.129450761432
  (1, 31123)	0.116775518268
  (1, 10302)	0.0797108439834
  (1, 7831)	0.176736539174
  (1, 43756)	0.0797108439834
  (1, 1885)	0.0838096269878
  (1, 5411)	0.0679026764305
  (1, 31732)	0.089084966057
  (1, 38273)	0.102735537169
  (1, 44046)	0.0800556182295
  (1, 45950)	0.0960768681014
  (1, 17635)	0.240989165277
  (1, 47144)	0.0874223538907
  (1, 14034)	0.0742547705572
  (1, 41046)	0.106732115752
  (1, 32781)	0.138885720703
  (1, 39241)	0.0787243392195
  (1, 24799)	0.064152586744
  (1, 31790)	0.0704076165584
  :	:
  (31109, 4269)	0.0377460871041
  (31109, 35272)	0.0577218965356
  (31109, 24686)	0.0417189988265
  (31109, 21468)	0.0281246360286
  (31109, 3707)	0.100935388312
  (31109, 14571)	0.0423268753826
  (31109, 46080)	0.0482403344871
  (31109, 11701)	0.0361762110722
  (31109, 2750)	0.0289037009

In [31]:
print(X_test)

  (0, 10272)	0.248899690119
  (0, 44883)	0.248899690119
  (0, 3670)	0.2658434338
  (0, 23632)	0.5316868676
  (0, 23979)	0.210811382518
  (0, 32119)	0.192613791953
  (0, 11749)	0.22008787539
  (0, 25430)	0.222044491763
  (0, 19132)	0.130824386178
  (0, 26435)	0.119020203111
  (0, 14463)	0.18765297304
  (0, 29282)	0.11753117282
  (0, 35354)	0.163801977225
  (0, 31704)	0.106775275246
  (0, 38178)	0.142796360154
  (0, 39919)	0.138101859336
  (0, 5235)	0.220350590333
  (0, 12337)	0.0936864965212
  (0, 15249)	0.12099438661
  (0, 45669)	0.102670656056
  (0, 25754)	0.0851553043744
  (0, 14683)	0.0885781407425
  (0, 11701)	0.0911815794986
  (0, 44489)	0.112900334261
  (0, 21584)	0.073057114076
  :	:
  (7776, 14571)	0.0995837462847
  (7776, 17076)	0.0685402501629
  (7776, 37387)	0.105277003827
  (7776, 30447)	0.0601834483687
  (7776, 40154)	0.0636014563273
  (7776, 26337)	0.284373096009
  (7776, 44750)	0.0794525675833
  (7776, 33974)	0.0831898056697
  (7776, 41667)	0.0833808194948
  (7776, 38357

In [102]:
def train_RF(n_est, depth):
    rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth, n_jobs=-1, verbose=10)
    rf_model = rf.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    precision, recall, fscore, support = score(y_test, y_pred)
    print('Est: {} / Depth: {} ---- Precision: {} / Recall: {} / Accuracy: {}'.format(
        n_est, depth, precision, recall,
        round((y_pred==y_test).sum() / len(y_pred), 3)))

In [14]:
from ipywidgets import FloatProgress
from IPython.display import display

In [20]:
max_count = 12
count = 0
progressBar = FloatProgress(min=0, max=max_count)
display(progressBar)

for n_est in [10, 50, 100]:
    for depth in [10, 20, 30, None]:
        print("\n\nTraining combination: n_est:{n_est} - depth:{max_depth}".format(n_est=str(n_est),depth=str(depth)))
        progressBar.value+=1
        count += 1
        train_RF(n_est, depth)

A Jupyter Widget



Training combination: n_est:10 - max_depth:10


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 10 / Depth: 10 ---- Precision: [ 0.48833034  0.          0.625       0.          0.          0.          0.64
  0.          0.          0.          0.          0.13306908  0.
  0.73333333  0.          0.          0.          0.          0.
  0.61538462  0.38461538  0.          0.76271186  0.          0.14427481
  0.59447005  0.7         0.          0.          0.          0.625       0.
  0.          0.          1.          0.33333333  0.38271605  0.
  0.68627451  0.          0.          0.          0.53218884  0.
  0.74285714  0.          0.          0.          0.          0.          0.
  0.32142857  0.95454545  0.5         0.          0.          0.
  0.26470588  0.5         0.61111111  0.          0.          0.
  0.28508772  0.          0.          0.          0.          0.
  0.66486486  0.73387097  0.          0.77272727  0.          0.41734417
  0.          0.          0.59183673  0.          0.          0.          0.
  0.          0.          0.55844156  0.          0. 

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


...

Est: 100 / Depth: 20 ---- Precision: [ 0.74100719  0.9375      0.69230769  1.          0.75        0.          0.
  0.          0.          0.          0.73809524  0.22864483  0.          1.
  0.          0.          0.83333333  0.          0.          0.65
  0.91935484  0.          0.8372093   0.          0.23338185  0.79411765
  0.75838926  0.8         0.          0.          0.8375      0.          0.
  0.80952381  0.6         1.          0.32773109  0.          0.77027027
  0.          0.          0.          0.49343832  0.          0.83333333
  0.63157895  0.          0.          0.          0.          0.
  0.54545455  0.          0.52173913  0.          0.5         0.          0.5034965
  0.86666667  0.63432836  0.          0.75        0.          0.32291667
  0.          0.          0.          0.          0.          0.88125
  0.92134831  0.          0.79746835  1.          0.58461538  0.          0.
  0.69026549  0.          0.          0.          0.          0.          0.


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 100 / Depth: 30 ---- Precision: [ 0.81315789  0.86666667  0.58333333  0.75        0.60526316  0.71428571
  0.          0.          0.          0.          0.7         0.27734375
  0.          0.91666667  0.          0.          0.94444444  0.          0.
  0.76470588  0.81578947  0.          0.81203008  1.          0.3423161
  0.78285714  0.66483516  0.875       1.          0.          0.85526316
  0.          0.          0.73333333  0.42857143  0.33333333  0.37223975
  0.          0.78205128  0.5         0.          0.          0.51378446
  0.          0.83333333  0.50666667  0.          0.          0.          0.
  0.          0.5         0.91666667  0.39344262  0.          0.57142857
  0.          0.47945205  0.8         0.64748201  0.          1.          0.
  0.32996633  1.          0.          0.          0.          0.
  0.81818182  0.87735849  0.          0.76190476  0.53333333  0.61621622
  0.          0.          0.74231678  0.          0.          0.          0.4
  0.  

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 100 / Depth: None ---- Precision: [ 0.90144928  0.84210526  0.53191489  0.7037037   0.55555556  0.57692308
  0.72727273  0.83333333  0.          0.          0.64383562  0.4         0.
  0.60869565  0.          0.          0.81967213  1.          0.06666667
  0.68539326  0.82142857  0.          0.77142857  0.875       0.60213618
  0.78494624  0.63768116  0.79545455  1.          0.          0.82954545
  0.18571429  0.4         0.74193548  0.5         0.51724138  0.37412587
  1.          0.625       0.44444444  0.5         0.          0.54061625
  0.          0.77083333  0.54081633  0.4         0.10416667  0.7         0.
  0.          0.37179487  0.95454545  0.37815126  0.          0.5         1.
  0.50980392  0.84615385  0.68055556  1.          0.92857143  0.          0.33
  0.9047619   0.14285714  0.          0.5         0.          0.87058824
  0.88695652  1.          0.78571429  0.37931034  0.69470405  1.
  0.36111111  0.78030303  0.          0.33333333  0.          0.45833333
  

In [31]:
train_RF(900, None) # 0.613 accuracy if i use wordnet lemamtizing

building tree 1 of 900
building tree 4 of 900
building tree 3 of 900
building tree 2 of 900
building tree 5 of 900
building tree 6 of 900
building tree 7 of 900
building tree 8 of 900


[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   11.2s


building tree 9 of 900
building tree 10 of 900
building tree 11 of 900
building tree 12 of 900
building tree 13 of 900
building tree 14 of 900
building tree 15 of 900
building tree 16 of 900

[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   17.6s



building tree 17 of 900
building tree 18 of 900
building tree 19 of 900
building tree 20 of 900
building tree 21 of 900


...

building tree 837 of 900
building tree 838 of 900
building tree 839 of 900
building tree 840 of 900
building tree 841 of 900
building tree 842 of 900
building tree 843 of 900
building tree 844 of 900
building tree 845 of 900
building tree 846 of 900
building tree 847 of 900
building tree 848 of 900
building tree 849 of 900
building tree 850 of 900
building tree 851 of 900
building tree 852 of 900
building tree 853 of 900
building tree 854 of 900
building tree 855 of 900
building tree 856 of 900
building tree 857 of 900
building tree 858 of 900
building tree 859 of 900
building tree 860 of 900
building tree 861 of 900
building tree 862 of 900
building tree 863 of 900
building tree 864 of 900
building tree 865 of 900
building tree 866 of 900
building tree 867 of 900
building tree 868 of 900
building tree 869 of 900
building tree 870 of 900
building tree 871 of 900
building tree 872 of 900
building tree 873 of 900
building tree 874 of 900
building tree 875 of 900
building tree 876 of 900


[Parallel(n_jobs=-1)]: Done 874 tasks      | elapsed: 28.2min


building tree 878 of 900
building tree 879 of 900
building tree 880 of 900
building tree 881 of 900
building tree 882 of 900
building tree 883 of 900
building tree 884 of 900
building tree 885 of 900
building tree 886 of 900
building tree 887 of 900
building tree 888 of 900
building tree 889 of 900
building tree 890 of 900
building tree 891 of 900
building tree 892 of 900
building tree 893 of 900
building tree 894 of 900
building tree 895 of 900
building tree 896 of 900
building tree 897 of 900
building tree 898 of 900
building tree 899 of 900
building tree 900 of 900


[Parallel(n_jobs=-1)]: Done 900 out of 900 | elapsed: 28.9min finished
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    1.3s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    1.8s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    1.9s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    2.4s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.1s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    6.5s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    6.9s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:    7.2s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:    7.4s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:    7.8s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:   11.0s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:   11.9s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   12.3s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   12.8s
[Parallel(n_jo

Est: 900 / Depth: None ---- Precision: [ 0.8617284   0.77777778  0.73214286  0.6         0.47826087  0.53731343
  0.6875      1.          0.          0.          0.75862069  0.45292621
  0.          0.73333333  0.          0.          0.89830508  0.375       0.15
  0.61333333  0.84313725  0.          0.86861314  0.875       0.5744089
  0.81656805  0.62264151  0.85106383  0.85714286  0.          0.79487179
  0.21052632  0.44444444  0.75862069  0.39285714  0.63636364  0.35534591
  1.          0.7826087   0.66666667  0.          0.          0.53250774
  0.          0.83333333  0.62765957  0.57142857  0.28        0.6
  0.22222222  0.          0.47142857  0.6875      0.38848921  0.
  0.57894737  0.66666667  0.51369863  0.8         0.62328767  1.          1.
  0.          0.37647059  0.8         0.12        0.          0.63636364
  0.          0.90116279  0.92682927  1.          0.91025641  0.52173913
  0.65373134  0.          0.21568627  0.77925532  0.          0.36363636
  0.          0.64

In [103]:
train_RF(300, None) # 0.618 accuracy if i use wordnet lemamtizing

building tree 1 of 300
building tree 3 of 300building tree 2 of 300building tree 4 of 300


building tree 5 of 300
building tree 6 of 300
building tree 7 of 300
building tree 8 of 300


[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   11.8s


building tree 9 of 300
building tree 10 of 300
building tree 11 of 300
building tree 12 of 300
building tree 13 of 300
building tree 14 of 300


[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   17.7s


building tree 15 of 300
building tree 16 of 300
building tree 17 of 300
building tree 18 of 300
building tree 19 of 300
building tree 20 of 300


...

building tree 261 of 300
building tree 262 of 300
building tree 263 of 300
building tree 264 of 300
building tree 265 of 300
building tree 266 of 300
building tree 267 of 300
building tree 268 of 300
building tree 269 of 300
building tree 270 of 300
building tree 271 of 300
building tree 272 of 300
building tree 273 of 300
building tree 274 of 300
building tree 275 of 300
building tree 276 of 300
building tree 277 of 300
building tree 278 of 300
building tree 279 of 300
building tree 280 of 300
building tree 281 of 300
building tree 282 of 300
building tree 283 of 300
building tree 284 of 300


[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:  7.6min


building tree 285 of 300
building tree 286 of 300
building tree 287 of 300
building tree 288 of 300
building tree 289 of 300
building tree 290 of 300
building tree 291 of 300
building tree 292 of 300
building tree 293 of 300
building tree 294 of 300
building tree 295 of 300
building tree 296 of 300
building tree 297 of 300
building tree 298 of 300
building tree 299 of 300
building tree 300 of 300


[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  8.2min finished
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.7s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    1.4s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    2.1s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    4.3s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    4.6s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    4.8s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    5.1s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:    6.9s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:   10.3s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:   10.8s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:   11.1s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:   13.2s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   14.5s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   15.1s
[Parallel(n_jo

Est: 300 / Depth: None ---- Precision: [ 0.92915531  0.83333333  0.59166667  0.83333333  0.35555556  0.61538462
  0.73684211  0.88888889  0.          0.5         0.75        0.44536817
  0.5         0.68181818  1.          0.89230769  0.8         0.
  0.67326733  0.86516854  0.          0.91603053  0.76923077  0.58474576
  0.83636364  0.56730769  0.86885246  0.92857143  0.          0.77319588
  0.25730994  0.71428571  0.84848485  0.525       0.64516129  0.28571429
  1.          0.71111111  0.57142857  0.66666667  1.          0.56923077
  0.          0.76923077  0.62765957  0.6         0.31764706  0.45454545
  0.33333333  0.          0.35714286  1.          0.43609023  0.          0.6
  0.75        0.47402597  0.7721519   0.72727273  0.8         0.73913043
  0.          0.41697417  0.875       0.21875     0.66666667  0.7         0.
  0.9         0.88118812  1.          0.82352941  0.52941176  0.58255452
  1.          0.22222222  0.79076087  1.          0.625       0.          0.56
  0.8

In [14]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier()
param = {'n_estimators': [10, 150, 300],
        'max_depth': [30, 90, None],
         'verbose': [10]
        }


In [ ]:
gs = GridSearchCV(rf, param, cv=5, n_jobs=-1, verbose=10)
gs_fit = gs.fit(tfidf_vectorized_matrix_ps, data['label'])
pd.DataFrame(gs_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

Fitting 5 folds for each of 9 candidates, totalling 45 fits


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\model_selection\_split.py:597: Warning: The least populated class in y has only 3 members, which is too few. The minimum number of members in any class cannot be less than n_splits=5.
  % (min_groups, self.n_splits)), Warning)
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   17.2s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:  4.2min
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:  8.6min
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed: 21.6min


### my laptop doesnt have enough memory to run GridsearchCV above 
### so I ran the same on EC2 instance and got n_est 300 and depth None as best mean test score

In [15]:
def train_GB(est, max_depth, lr):
    gb = GradientBoostingClassifier(n_estimators=est, max_depth=max_depth, learning_rate=lr , verbose=10)
    gb_model = gb.fit(X_train, y_train)
    y_pred = gb_model.predict(X_test)
    precision, recall, fscore, train_support = score(y_test, y_pred)
    print('Est: {} / Depth: {} / LR: {} ---- Precision: {} / Recall: {} / Accuracy: {}'.format(
        est, max_depth, lr, precision,recall, 
        round((y_pred==y_test).sum()/len(y_pred), 3)))

In [16]:
from ipywidgets import FloatProgress
from IPython.display import display




In [ ]:
max_count = 12
count = 0
progressBar = FloatProgress(min=0, max=max_count)
display(progressBar)

for n_est in [50, 100, 150]:
    for max_depth in [3, 7, 11, 15]:
        print("\n\nTraining combination: n_est:{n_est} - max_depth:{max_depth}".format(n_est=str(n_est),max_depth=str(max_depth)))
        progressBar.value+=1
        count += 1
        train_GB(n_est, max_depth, 0.01)

A Jupyter Widget



Training combination: 50 - 3


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples.
  'recall', 'true', average, warn_for)


Est: 50 / Depth: 3 / LR: 0.01 ---- Precision: [ 0.93452381  0.96        0.73451327  0.75        0.28        0.48        0.71875
  0.45        0.          0.41666667  0.81538462  0.34558824  0.
  0.72972973  0.          0.16666667  0.74545455  0.83333333  0.07692308
  0.61458333  0.83333333  0.          0.90833333  0.75        0.65492958
  0.81761006  0.7         0.69565217  0.76470588  0.5         0.81428571
  0.30508475  0.57142857  0.90243902  0.53125     0.53846154  0.28851541
  0.66666667  0.65137615  0.38461538  0.83333333  1.          0.49832776
  0.          0.86956522  0.43434343  0.66666667  0.19354839  0.66666667
  0.08333333  0.55555556  0.40243902  1.          0.4159292   0.          0.53125
  1.          0.55555556  0.81578947  0.8         0.53846154  0.76470588
  0.          0.36394558  0.84615385  0.11363636  0.75        0.59259259
  0.          0.91156463  0.8877551   0.          0.89041096  0.41176471
  0.68543046  0.625       0.18681319  0.77777778  0.          0.6666

Est: 50 / Depth: 15 / LR: 0.01 ---- Precision: [ 0.94864048  0.95238095  0.62601626  0.59375     0.34285714  0.49019608
  0.64        0.64285714  0.          0.          0.77777778  0.27158099
  0.          0.64516129  0.          0.2         0.84444444  0.77777778
  0.03846154  0.52991453  0.79710145  0.          0.90677966  0.8
  0.66793893  0.82550336  0.69871795  0.70212766  0.92857143  0.
  0.81355932  0.17682927  0.5         0.92105263  0.48275862  0.46666667
  0.31481481  0.42857143  0.6         0.30434783  0.83333333  1.          0.4924812
  0.          0.82978723  0.44615385  0.52941176  0.09174312  0.42307692
  0.25        0.14814815  0.32467532  1.          0.37162162  0.
  0.51612903  1.          0.56521739  0.8358209   0.77777778  0.4
  0.90909091  0.          0.37089202  0.8         0.08823529  0.5
  0.45714286  0.07692308  0.93377483  0.91304348  0.25        0.92647059
  0.44444444  0.67054264  0.71428571  0.15789474  0.78456592  0.          0.6
  0.          0.37209302 

In [ ]:
gb = GradientBoostingClassifier()
param = {
    'n_estimators': [50, 100, 150], 
    'max_depth': [7, 11, 15],
    'learning_rate': [0.01]
}

clf = GridSearchCV(gb, param, cv=5, n_jobs=-1)
cv_fit = clf.fit(tfidf_vectorized_matrix_ps, data['label'])
pd.DataFrame(cv_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

In [26]:
tfidf_vect_fit = tfidf_vectorizer_ps.fit(data['feature'])

tfidf_train = tfidf_vect_fit.transform(data['feature'])


In [27]:
rf = RandomForestClassifier(n_estimators=300, max_depth=None, n_jobs=-1, verbose=10)
rf_model = rf.fit(tfidf_train, data['label'])

building tree 1 of 300building tree 2 of 300building tree 3 of 300
building tree 4 of 300


building tree 5 of 300
building tree 6 of 300
building tree 7 of 300
building tree 8 of 300


[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   21.2s


building tree 9 of 300
building tree 10 of 300
building tree 11 of 300
building tree 12 of 300
building tree 13 of 300
building tree 14 of 300
building tree 15 of 300
building tree 16 of 300


[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   31.2s


building tree 17 of 300
building tree 18 of 300
building tree 19 of 300
building tree 20 of 300
building tree 21 of 300


...

[Parallel(n_jobs=-1)]: Done 257 tasks      | elapsed: 11.9min


building tree 262 of 300
building tree 263 of 300
building tree 264 of 300
building tree 265 of 300
building tree 266 of 300
building tree 267 of 300
building tree 268 of 300
building tree 269 of 300
building tree 270 of 300
building tree 271 of 300
building tree 272 of 300
building tree 273 of 300
building tree 274 of 300
building tree 275 of 300
building tree 276 of 300
building tree 277 of 300
building tree 278 of 300
building tree 279 of 300
building tree 280 of 300
building tree 281 of 300
building tree 282 of 300
building tree 283 of 300


[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed: 12.7min


building tree 284 of 300
building tree 285 of 300
building tree 286 of 300
building tree 287 of 300
building tree 288 of 300
building tree 289 of 300
building tree 290 of 300
building tree 291 of 300
building tree 292 of 300
building tree 293 of 300
building tree 294 of 300
building tree 295 of 300
building tree 296 of 300
building tree 297 of 300
building tree 298 of 300
building tree 299 of 300
building tree 300 of 300


[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed: 13.5min finished


In [48]:
list(zip(tfidf_vectorizer_ps.get_feature_names(),rf_model.feature_importances_))
#print(type(tfidf_vectorizer_ps.get_feature_names()))
#print(type(rf_model.feature_importances_))
features_importances_df=pd.DataFrame(list(zip(tfidf_vectorizer_ps.get_feature_names(),rf_model.feature_importances_)))
features_importances_df.columns = ['feature','importance']

In [118]:
features_importances_df['match']=features_importances_df['feature'].str.match('^[a-zA-Z]+$')
features_importances_df[features_importances_df['match']==True  ][['feature']]

,feature
1412,a
1451,aa
1452,aaa
1454,aaaa
1455,aaaaaa
1456,aaaaaaaargh
1457,aaabc
1458,aaabsolut
1459,aaacc
1460,aaactiv


In [123]:
top_features_df=features_importances_df.sort_values('importance',ascending =False)[0:40000]
top_features_df[top_features_df['match']==True  ][['feature']].to_csv('features_importances.csv',index=False)
top_features_df

,feature,importance,match
9155,clean,1.186558e-02,True
33032,plumb,8.892218e-03,True
38446,servic,7.519486e-03,True
1694,account,7.503087e-03,True
32566,photographi,5.551820e-03,True
14276,electr,5.514186e-03,True
6336,bookkeep,5.372179e-03,True
42975,tile,5.328983e-03,True
7385,busi,5.209701e-03,True
35938,renov,4.958713e-03,True


In [92]:
#features_importances_df.head()
#print(features_importances_df.sort_values('importance',ascending =False)[0:40000])

import_features_list=list(features_importances_df.sort_values('importance',ascending =False)[0:40000]['feature'])
import_features_list

['clean',
 'plumb',
 'servic',
 'account',
 'photographi',
 'electr',
 'bookkeep',
 'tile',
 'busi',
 'renov',
 'concret',
 'paint',
 'build',
 'provid',
 'work',
 'garden',
 'home',
 'experi',
 'profession',
 'comput',
 'qualiti',
 'remov',
 'year',
 'job',
 'fenc',
 'design',
 'pest',
 'specialis',
 'tax',
 'small',
 'photograph',
 'mainten',
 'aspect',
 'repair',
 'construct',
 'need',
 'roof',
 'commerci',
 'celebr',
 'offer',
 'landscap',
 'carpet',
 'cater',
 'ga',
 'wed',
 'move',
 'industri',
 'window',
 'compani',
 'train',
 'fit',
 'bathroom',
 'best',
 'high',
 'detail',
 'custom',
 'makeup',
 'floor',
 'team',
 'includ',
 'new',
 'mechan',
 'reliabl',
 'instal',
 'control',
 'free',
 'mow',
 'tree',
 'lawn',
 'also',
 'ceremoni',
 'person',
 'big',
 'pti',
 'domest',
 'skill',
 'base',
 'ltd',
 'dog',
 'project',
 'client',
 'plaster',
 'rang',
 'air',
 'quot',
 'area',
 'carpentri',
 'solut',
 'massag',
 'call',
 'car',
 'dj',
 'portrait',
 'complet',
 'us',
 'offic',
 'pr

In [20]:
QUERY = """ 
SELECT
bm.id
,trim(nvl(bm.name,'') || ' ' || nvl(description,'')) "feature"
FROM oneflare_reports.business_master bm
WHERE bm.total_quotes_made < 3
"""

In [21]:
data_to_predict=pd.read_sql(QUERY,redshift)

In [22]:
data_to_predict.head()

,id,feature
0,94,Royal Flush Flooring
1,113,Barwon Electrical Services Barwon Electrical Services is a friendly Electrical business with a w...
2,231,Blitz Floors And Blinds Blitz Blinds is a family owned local business which has been established...
3,235,Edwards Interiors Domestic & Commercial Supply & installation of\r\n*Plasterboard & Cornices\r\n...
4,333,Zac's Home Insulation Zac's Home Insulation is an Australian owned and operated company providin...


In [28]:
#tfidf_vectorizer_ps = TfidfVectorizer(analyzer=clean_text_by_porter_stem)
tfidf_vectorized_matrix_to_predict_ps = tfidf_vect_fit.transform(data_to_predict['feature'] )

In [29]:
y_to_predict = rf_model.predict(tfidf_vectorized_matrix_to_predict_ps)

[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.7s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    1.2s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    1.7s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    2.4s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.4s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    6.4s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    9.7s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:   12.0s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:   12.9s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:   15.7s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:   18.5s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:   21.0s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   24.3s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   26.6s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:   30.3s
[Parallel(n_jobs=4)]: Do

In [30]:
data_to_predict['predited_category'] = y_to_predict

In [31]:
data_to_predict.to_csv('predited_category.csv',index=False)

In [34]:
pd.set_option('display.max_colwidth', -1)
data_to_predict.head()

,id,feature,predited_category
0,94,Royal Flush Flooring,Concreting
1,113,Barwon Electrical Services Barwon Electrical Services is a friendly Electrical business with a wealth of experience that can cater for all your commercial and domestic electrical needs.\nCover all aspects of electrical from new homes and developments through to smaller jobs and general maintenance.\nOur professional and courteous service ensures you that all our work is completed to the highest standard at affordable rates.,Home Appliance Repair
2,231,"Blitz Floors And Blinds Blitz Blinds is a family owned local business which has been established for the past 15 years servicing the Western suburbs of Victoria with quality blinds. \r\n\r\nWe specialise in the supply and installation of quality custom made to measure blinds and curtains.\r\n\r\nBlitz Floors and Blinds offer the ultimate in quality and style for all your Drapes, Pelmets, Holland, E- screen, Shutters, Romans, Timber, Venetians and Swags and Tails.\r\n\r\nTo build a successful business, you need to understand how important existing customers are and the importance of creating new long term customers. \r\nWe strive to offer the highest levels of quality products & commitment to our customers with the best service, workmanship and price guaranteed.\r\n\r\nWe provide our customers with:\r\nFree Measure and quote\r\nFree Consultation\r\nWe come to you\r\nExpert Advice\r\nServicing all Suburbs",Flooring
3,235,Edwards Interiors Domestic & Commercial Supply & installation of\r\n*Plasterboard & Cornices\r\n*Suspended Ceilings\r\n*Partition Systems\r\n*Fire rated Systems & Insulation\r\n*Demolition & Waste removal,Rubbish Removal
4,333,Zac's Home Insulation Zac's Home Insulation is an Australian owned and operated company providing high quality insulation for your home or business.,Builder


In [32]:
from sklearn.externals import joblib
joblib.dump(rf_model, 'random_forest.pkl') 
#clf = joblib.load('filename.pk1')

['random_forest.pkl']

In [35]:
len(data_to_predict)

33373